# RAG Pipeline with LangChain + FAISS

This notebook builds a complete Retrieval-Augmented Generation (RAG) pipeline for:
- `Events.csv`
- `Faculty.xlsx`
- `PlacementGuide.pdf`
- `Rulebook.pdf`
- `Timetable.xlsx`

**Pipeline:** Load files → Chunk text → Generate embeddings → Store in FAISS vector DB → Save as `.faiss` + `.pkl` → Query with natural language.

> Run cells top to bottom. If a package is missing, the install cell below handles it.

## 1. Install dependencies

In [8]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu \
    sentence-transformers pypdf "unstructured[xlsx]" openpyxl pandas tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 3.1 MB/s eta 0:00:00


## 2. Imports

In [9]:
import os
import glob

from langchain_community.document_loaders import (
    CSVLoader,
    UnstructuredExcelLoader,
    PyPDFLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

## 3. Configure file paths

Edit `DATA_DIR` if your files live elsewhere (e.g. `/content/drive/MyDrive/...` if mounted from Google Drive).

In [10]:
DATA_DIR = "/content"  # folder containing your files in Colab

FILES = {
    "csv": ["Events.csv"],
    "xlsx": ["Faculty.xlsx", "Timetable.xlsx"],
    "pdf": ["PlacementGuide.pdf", "Rulebook.pdf"],
}

def full_path(fname):
    return os.path.join(DATA_DIR, fname)

for category, names in FILES.items():
    for n in names:
        p = full_path(n)
        print(f"{'FOUND   ' if os.path.exists(p) else 'MISSING '} {p}")

FOUND    /content/Events.csv
FOUND    /content/Faculty.xlsx
FOUND    /content/Timetable.xlsx
FOUND    /content/PlacementGuide.pdf
FOUND    /content/Rulebook.pdf


## 4. Load all documents

Each loader tags every chunk-source with metadata (`source`, `file_type`) so you can trace answers back to the original file.

In [11]:
all_docs = []

def load_and_tag(loader, source_name, file_type):
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = source_name
        d.metadata["file_type"] = file_type
    return docs

# --- CSV files ---
for fname in FILES["csv"]:
    path = full_path(fname)
    if os.path.exists(path):
        loader = CSVLoader(file_path=path, encoding="utf-8")
        all_docs.extend(load_and_tag(loader, fname, "csv"))

# --- Excel files ---
for fname in FILES["xlsx"]:
    path = full_path(fname)
    if os.path.exists(path):
        loader = UnstructuredExcelLoader(path, mode="elements")
        all_docs.extend(load_and_tag(loader, fname, "xlsx"))

# --- PDF files ---
for fname in FILES["pdf"]:
    path = full_path(fname)
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        all_docs.extend(load_and_tag(loader, fname, "pdf"))

print(f"Total documents loaded: {len(all_docs)}")

Total documents loaded: 10


## 5. Chunk the documents

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")
print("\nSample chunk:\n", chunks[0].page_content[:300] if chunks else "No chunks")
print("\nMetadata:\n", chunks[0].metadata if chunks else "")

Total chunks created: 11

Sample chunk:
 Event: AI Workshop
Date: 12-Aug-2026
Time: 10:00 AM
Venue: Seminar Hall
Coordinator: Dr. Ravi

Metadata:
 {'source': 'Events.csv', 'row': 0, 'file_type': 'csv'}


## 6. Generate embeddings

Using a free, local `sentence-transformers` model — no API key required. Swap in `OpenAIEmbeddings()` if you prefer OpenAI (needs `OPENAI_API_KEY`).

In [13]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# --- Alternative: OpenAI embeddings ---
# import os
# os.environ["OPENAI_API_KEY"] = "your-key-here"
# from langchain_openai import OpenAIEmbeddings
# embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 7. Build the FAISS vector store

In [14]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
print("Vector store created with", vectorstore.index.ntotal, "vectors")

Vector store created with 11 vectors


## 8. Save the vector store as `.faiss` + `.pkl`

`FAISS.save_local()` writes two files into the target folder:
- `index.faiss` — the vector index
- `index.pkl` — the docstore + metadata mapping

In [15]:
SAVE_DIR = "/content/vector_store"
os.makedirs(SAVE_DIR, exist_ok=True)

vectorstore.save_local(SAVE_DIR, index_name="index")

print("Saved files:")
for f in os.listdir(SAVE_DIR):
    print(" -", os.path.join(SAVE_DIR, f))

Saved files:
 - /content/vector_store/index.pkl
 - /content/vector_store/index.faiss


## 9. Reload the vector store from disk (sanity check)

In [16]:
loaded_vectorstore = FAISS.load_local(
    SAVE_DIR,
    embedding_model,
    index_name="index",
    allow_dangerous_deserialization=True,
)
print("Reloaded vector store with", loaded_vectorstore.index.ntotal, "vectors")

Reloaded vector store with 11 vectors


## 10. Query the vector store

Simple similarity search — returns the most relevant chunks with their source file.

In [17]:
def search(query, k=4):
    results = loaded_vectorstore.similarity_search_with_score(query, k=k)
    for i, (doc, score) in enumerate(results, 1):
        print(f"\n--- Result {i} (score={score:.4f}, source={doc.metadata.get('source')}) ---")
        print(doc.page_content[:400])
    return results

# Example query
_ = search("When is the placement drive and what are the eligibility rules?")


--- Result 1 (score=1.0401, source=PlacementGuide.pdf) ---
Campus Placement Handbook 2026
Eligibility
– Minimum CGPA: 7.0
– No active backlogs.
– Minimum attendance: 75%
Placement Process
– Registration
– Resume Verification
– Aptitude Test
– Technical Interview
– HR Interview
– Offer Letter
Required Documents
– Resume
– Aadhaar Card
– PAN Card
– Semester Marks Cards
– Passport Size Photograph
Resume Guidelines
– Maximum 2 pages
– Mention projects
– Menti

--- Result 2 (score=1.0785, source=Events.csv) ---
Event: Placement Drive
Date: 30-Aug-2026
Time: 9:00 AM
Venue: Placement Cell
Coordinator: Dr. Vinay

--- Result 3 (score=1.1985, source=Rulebook.pdf) ---
Student Academic Rulebook 2026
Attendance Rules
– Minimum attendance required: 75%
– Students below 75% attendance are not eligible to appear for semester examinations un-
less approved by the Principal.
– Attendance is calculated separately for every subject.
Examination Rules
– Mid Semester Exam: 30 Marks
– End Semester Exam: 70 

## 11. (Optional) Full RAG answer with an LLM

Wraps retrieval + generation together. Requires an LLM — example uses OpenAI; swap in any LangChain-supported chat model (e.g. `ChatGoogleGenerativeAI`, `ChatAnthropic`, local Ollama model, etc.).

In [18]:
# import os
# os.environ["OPENAI_API_KEY"] = "your-key-here"
# from langchain_openai import ChatOpenAI
# from langchain.chains import RetrievalQA
#
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     retriever=loaded_vectorstore.as_retriever(search_kwargs={"k": 4}),
#     return_source_documents=True,
# )
#
# response = qa_chain.invoke({"query": "What is the college timetable for Monday?"})
# print(response["result"])
# print("\nSources:", [d.metadata["source"] for d in response["source_documents"]])

## 12. Interactive query loop (Colab)

Run this cell and type questions; type `exit` to stop.

In [19]:
while True:
    q = input("\nAsk a question (or 'exit'): ")
    if q.strip().lower() == "exit":
        break
    search(q)

KeyboardInterrupt: Interrupted by user